## Loading imports and data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from FPCA import FPCA, Phimaker, omegacomputer, penaltymatrixmaker
import pandas as pd
import matplotlib.dates as mdates

In [ ]:
def partmissing(sensorseries):
    mask = ~np.isnan(sensorseries)
    if not mask.any(): return 100
    start_idx = np.argmax(mask)
    end_idx = len(sensorseries) - np.argmax(mask[::-1]) - 1
    series = sensorseries[start_idx:end_idx+1]
    return np.isnan(series).sum()/len(series) * 100

def partoutlier(sensorseries):
    mask = ~np.isnan(sensorseries)
    if not mask.any(): return None
    start_idx = np.argmax(mask)
    end_idx = len(sensorseries) - np.argmax(mask[::-1]) - 1
    series = sensorseries[start_idx:end_idx+1]

    nonmissing = np.sum(~np.isnan(series))
    outlier_count = np.sum(series > 500)
    return outlier_count / nonmissing * 100

def get_start_percentage(sensorseries):
    mask = ~np.isnan(sensorseries)
    start_idx = np.argmax(mask)
    return (start_idx / len(sensorseries)) * 100

def longest_high_run(sensorseries, threshold=500):
    mask = ~np.isnan(sensorseries)
    if not mask.any(): return None
    start_idx = np.argmax(mask)
    end_idx = len(sensorseries) - np.argmax(mask[::-1]) - 1
    series = sensorseries[start_idx:end_idx+1]

    high = series > threshold

    max_run = 0
    current = 0
    for x in high:
        if x:
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0
    return max_run

def dataloader():
    raw_pollution = ...
    sensors = ...
    
    # Step 1: throw away completely useless sensors
    # Keep only viable sensors
    missing_rates = raw_pollution.apply(partmissing, axis=1)
    sensors['missing_rate'] = sensors.index.map(missing_rates)
    sensors = sensors[sensors['missing_rate'] < 50]
    raw_pollution = raw_pollution.loc[sensors.index]

    # Removing extreme outlier cases
    outlier_rates = raw_pollution.apply(partoutlier, axis=1)
    sensors['outlier_rate'] = sensors.index.map(outlier_rates)
    sensors = sensors[sensors['outlier_rate'] < 25]
    raw_pollution = raw_pollution.loc[sensors.index]

    # Removing consistent outlier sensors
    runs = raw_pollution.apply(longest_high_run, axis=1)
    sensors["max_high_run"] = sensors.index.map(runs)
    sensors = sensors[sensors.max_high_run < 14]
    raw_pollution = raw_pollution.loc[sensors.index]

    # Step 2: retrieving data that is viable for evaluating, but not usefull for training
    # In practice we do not use this, but might be usefull for other applications
    start_percentages = raw_pollution.apply(get_start_percentage, axis=1)
    sensors['start_at_percentage'] = raw_pollution.index.map(start_percentages)
    sensors_extra = sensors[sensors['start_at_percentage'] >= 50]
    sensors = sensors[sensors['start_at_percentage'] < 50]
    pollution_extra = raw_pollution.loc[sensors_extra.index]
    pollution = raw_pollution.loc[sensors.index]

    # Step 3: collecting all data
    dates = pd.to_datetime(raw_pollution.columns)
    longitude_extra = sensors_extra["longitude"].values
    latitude_extra = sensors_extra["latitude"].values
    pollution_extra = pollution_extra.values.T
    longitude = sensors["longitude"].values
    latitude = sensors['latitude'].values
    pollution = pollution.values.T
    pollution[pollution == 0] = np.nan

    return dates, longitude, latitude, pollution, longitude_extra, latitude_extra, pollution_extra

## Model specification

In [ ]:
def pooledgammaestimation(Y_list, Phi, R, l=0.01):
    ystar = []
    Phistar = []
    for y in Y_list:
        mask = ~np.isnan(y)
        y_valid = y[mask]
        ystar.append(y_valid)
        Phi_valid = Phi[mask]
        Phistar.append(Phi_valid)

    ystar = np.concatenate(ystar)
    Phistar = np.concatenate(Phistar)

    if Phi.shape[1] < 49:
        gammatilde = np.linalg.lstsq(Phistar, ystar, rcond=None)[0]
    else:
        gammatilde = np.linalg.solve(Phistar.T @ Phistar + l * R, Phistar.T @ ystar)
    return gammatilde

def regularCV(Y_list_train, Phi, R):
    L_candidates = np.array([0.0001, 0.001, 0.01, 0.1, 1])
    MAE = np.zeros_like(L_candidates)
    for i, l in enumerate(L_candidates):
        Mutilde_train_list, _ = onestepforecaster(Y_list_train[:20], Y_list_train[20:], Phi, Phi, R, l)
        MAE[i] = np.nanmean(np.abs(Y_list_train[20:] - Mutilde_train_list))

    return L_candidates[np.argmin(MAE)]

def onestepforecaster(Y_list_train, Y_list_test, Phi, Phi_surface, R, l, lags=20):
    N, K = Phi.shape
    Gamma_forecast = np.zeros((len(Y_list_test), K))
    history = list(Y_list_train[-lags:])
    for i in range(len(Y_list_test)):
        gammahat = pooledgammaestimation(history[-lags:], Phi, R, l)
        Gamma_forecast[i] = gammahat
        history.append(Y_list_test[i])

    return Gamma_forecast @ Phi.T, Gamma_forecast @ Phi_surface.T

## Performing FPCA

In [ ]:
def rescaler(x, eps = 0.01): return eps + (1 - 2*eps) * (x - np.min(x)) / (np.max(x) - np.min(x))

In [ ]:
dates, longitude, latitude, pollution, _, _, _ = dataloader()
eval_indices = np.arange(len(longitude)-75, len(longitude))
train_indices = np.arange(len(longitude)-75)
s1 = rescaler(longitude[train_indices])
s2 = rescaler(latitude[train_indices])
s1_eval = rescaler(longitude[eval_indices])
s2_eval = rescaler(latitude[eval_indices])
Phi = Phimaker(s1, s2)
Phi_surface = Phimaker(s1_eval, s2_eval)

Omega0, Omega1, Omega2 = omegacomputer()
R = penaltymatrixmaker(Omega0, Omega1, Omega2)

Y_list_surface = np.log(pollution[:, eval_indices])
Y_list = np.log(pollution[:, train_indices])
Y_list_train = Y_list[:-365]
Y_list_test = Y_list[-365:]


T, N = Y_list.shape
r = 7
_, Phi_eig, Phi_surface_eig, Level, Level_eval = FPCA(Y_list_train, Phi, Phi_surface, r)
Level_list = np.tile(Level, (T,1))
Level_list_eval = np.tile(Level_eval, (T,1))
Y_list = Y_list - Level_list
Y_list_surface = Y_list_surface - Level_list_eval
Y_list_train = Y_list[:-365]
Y_list_test = Y_list[-365:]
Y_list_surface_test = Y_list_surface[-365:]

## Model fit: Dimensionality reduction

In [ ]:
lstar = None 
Mutilde_list_test, Mutilde_list_surface_test = onestepforecaster(Y_list_train, Y_list_test, Phi_eig, Phi_surface_eig, R, lstar)
Mutilde_list_train, Mutilde_list_surface_train = onestepforecaster(Y_list_train[:20], Y_list_train[20:], Phi_eig, Phi_surface_eig, R, lstar)


Mutilde_list = np.concatenate((Mutilde_list_train, Mutilde_list_test)) + Level_list[20:]
Mutilde_list_surface = np.concatenate((Mutilde_list_surface_train, Mutilde_list_surface_test)) + Level_list_eval[20:]
Mutilde_list_complete_fpca = np.exp(np.hstack((Mutilde_list, Mutilde_list_surface)))

errors = np.abs(
    np.exp(Mutilde_list_test + Level_list[-365:])
    - np.exp(Y_list_test + Level_list[-365:])
)
daily_mae = np.nanmean(errors, axis=1)
MAE = np.mean(daily_mae)

errors_surface = np.abs(
    np.exp(Mutilde_list_surface_test + Level_list_eval[-365:])
    - np.exp(Y_list_surface_test + Level_list_eval[-365:])
)
daily_mae_surface = np.nanmean(errors_surface, axis=1)
MAE_surface = np.mean(daily_mae_surface)

print(f"Out of sample MAE of used sensors: {MAE}")
print(f"Out of sample MAE of unused sensors: {MAE_surface}")

plot_dict = {
    21 : "Milan, Italy (estimation sensor)", 
    62 : "Aegina, Greece (estimation sensor)", 
    164 : "Ljubljana, Slovenia (evaluation sensor)", 
    181 : "Hastings, United Kingdom (evaluation sensor)"}


for idx in plot_dict.keys():
    plt.figure(figsize=(10, 5))
    plt.plot(dates[20:], pollution[20:, idx], label=r"$y$", color='#7f7f7f')
    plt.plot(dates[20:], Mutilde_list_complete_fpca[:, idx], label=r"$\tilde{\mu}_{\text{t-GAS}}$", color='blue', alpha=0.6)
    plt.axvline(x=dates[-365], color='black', linestyle=':', linewidth=2, label="Train/Test Split")
    plt.xlim(dates[0], dates[-1])
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=10))
    plt.legend(fontsize=17)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)
    plt.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.3)
    plt.title(f"{plot_dict[idx]}", fontsize=22)
    plt.show()

## Model fit: No dimensionality reduction

In [ ]:
lstar = regularCV(Y_list_train, Phi, R)

Mutilde_list_test, Mutilde_list_surface_test = onestepforecaster(Y_list_train, Y_list_test, Phi, Phi_surface, R, l=lstar)
Mutilde_list_train, Mutilde_list_surface_train = onestepforecaster(Y_list_train[:20], Y_list_train[20:], Phi, Phi_surface, R, l=lstar)

Mutilde_list = np.concatenate((Mutilde_list_train, Mutilde_list_test)) + Level_list[20:]
Mutilde_list_surface = np.concatenate((Mutilde_list_surface_train, Mutilde_list_surface_test)) + Level_list_eval[20:]
Mutilde_list_complete_bspline = np.exp(np.hstack((Mutilde_list, Mutilde_list_surface)))


errors = np.abs(
    np.exp(Mutilde_list_test + Level_list[-365:])
    - np.exp(Y_list_test + Level_list[-365:])
)
daily_mae = np.nanmean(errors, axis=1)
MAE = np.mean(daily_mae)

errors_surface = np.abs(
    np.exp(Mutilde_list_surface_test + Level_list_eval[-365:])
    - np.exp(Y_list_surface_test + Level_list_eval[-365:])
)
daily_mae_surface = np.nanmean(errors_surface, axis=1)
MAE_surface = np.mean(daily_mae_surface)

print(f"Out of sample MAE of used sensors: {MAE}")
print(f"Out of sample MAE of unused sensors: {MAE_surface}")

plot_dict = {
    21 : "Milan, Italy (estimation sensor)", 
    62 : "Aegina, Greece (estimation sensor)", 
    164 : "Ljubljana, Slovenia (evaluation sensor)", 
    181 : "Hastings, United Kingdom (evaluation sensor)"}


for idx in plot_dict.keys():
    plt.figure(figsize=(10, 5))
    plt.plot(dates[20:], pollution[20:, idx], label=r"$y$", color='#7f7f7f')
    plt.plot(dates[20:], Mutilde_list_complete_bspline[:, idx], label=r"$\tilde{\mu}_{\text{pool, B-spline}}$", color='blue', alpha=0.35)
    plt.plot(dates[20:], Mutilde_list_complete_fpca[:, idx], label=r"$\tilde{\mu}_{\text{pool, FPCA}}$", color='red', alpha=0.35)
    plt.axvline(x=dates[-365], color='black', linestyle=':', linewidth=2, label="Train/Test Split")
    plt.xlim(dates[0], dates[-1])
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=10))
    plt.legend(fontsize=17)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)
    plt.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.3)
    plt.title(f"{plot_dict[idx]}", fontsize=22)
    plt.show()